# Cross-Validation with Over/Undersampling
## Joint Sentiment & Intent Classification — Sinhala NLP

**Strategy:**
- Stratified K-Fold on a **composite label** (sentiment × intent) to preserve the joint distribution across all folds.
- Per-fold: sampling is applied to the **training indices only** — the val fold is never touched (prevents data leakage).
- Reports per-fold and aggregate metrics for sentiment, intent, and joint accuracy.

**Sampling modes** (set via `SAMPLING_STRATEGY`):
| Value | Behaviour |
|---|---|
| `"oversample"` | RandomOverSampler — duplicate minority composite-label rows |
| `"undersample"` | RandomUnderSampler — drop majority rows |
| `"both"` | Undersample majority to 3× minority, then oversample minority up |
| `"none"` | No resampling (baseline) |

> **Requires:** `pip install imbalanced-learn`

## 0 — Install Dependencies

In [1]:
!pip install imbalanced-learn -q

## 1 — Imports

In [2]:
import os, re, unicodedata, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from tqdm import tqdm

print("All imports OK")

All imports OK


## 2 — Configuration

Edit `SAMPLING_STRATEGY` and `N_FOLDS` here before running.

In [3]:
# ── Paths ─────────────────────────────────────────────────────────────────────
FULL_CSV = "/kaggle/input/datasets/thiwankachanditha/sold-data-sentiment-intent/sinhala_silver_labels.csv"

# ── Model ─────────────────────────────────────────────────────────────────────
BACKBONE     = "xlm-roberta-base"
MAX_LEN      = 128
BATCH_SIZE   = 32
DROPOUT      = 0.3

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS       = 3       # reduced per-fold; early stopping will catch anyway
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP    = 1.0
ALPHA        = 0.5     # weight between sentiment loss and intent loss
PATIENCE     = 3
SEED         = 42
SENT_SMOOTH  = 0.10
INTENT_SMOOTH = 0.15

# ── Cross-validation & sampling ───────────────────────────────────────────────
N_FOLDS           = 2
SAMPLING_STRATEGY = "oversample"   # "oversample" | "undersample" | "both" | "none"

# ── Labels ────────────────────────────────────────────────────────────────────
SENTIMENT_CLASSES = ["positive", "negative", "neutral", "conflict"]
INTENT_CLASSES    = ["suggestion", "complaint", "inquiry", "feedback", "greeting", "other"]
NUM_SENTIMENTS    = len(SENTIMENT_CLASSES)
NUM_INTENTS       = len(INTENT_CLASSES)

SENTIMENT_MAP = {
    "positive": 0, "Positive": 0,
    "negative": 1, "Negative": 1,
    "neutral":  2, "Neutral":  2,
    "conflict": 3, "Conflict": 3,
}
INTENT_MAP = {
    "inquiry": 0,   "Inquiry": 0,
    "complaint": 1, "Complaint": 1,
    "suggestion": 2,"Suggestion": 2,
    "feedback": 3,  "Feedback": 3,
    "greeting": 4,  "Greeting": 4,
    "other": 5,     "Other": 5,
}

TEXT_COL      = "text"
SENTIMENT_COL = "sentiment_label"
INTENT_COL    = "intent_label"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Sampling strategy: {SAMPLING_STRATEGY}  |  Folds: {N_FOLDS}")

Device: cuda
Sampling strategy: oversample  |  Folds: 2


## 3 — Reproducibility

In [4]:
def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()
print(f"Seed set to {SEED}")

Seed set to 42


## 4 — Data Loading & Preprocessing

In [5]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def load_and_clean(path):
    df = pd.read_csv(path)
    df.rename(columns={"Sentence": "text", "Sentiment": "sentiment_label"}, inplace=True)

    # Warn about any label values not in our maps
    unknown_s = set(df[SENTIMENT_COL].unique()) - set(SENTIMENT_MAP)
    unknown_i = set(df[INTENT_COL].unique())   - set(INTENT_MAP)
    if unknown_s: print("WARNING – unknown sentiment values:", unknown_s)
    if unknown_i: print("WARNING – unknown intent values:",   unknown_i)

    df[SENTIMENT_COL] = df[SENTIMENT_COL].map(SENTIMENT_MAP)
    df[INTENT_COL]    = df[INTENT_COL].map(INTENT_MAP)
    before = len(df)
    df = df.dropna(subset=[SENTIMENT_COL, INTENT_COL])
    df[SENTIMENT_COL] = df[SENTIMENT_COL].astype(int)
    df[INTENT_COL]    = df[INTENT_COL].astype(int)

    df[TEXT_COL] = df[TEXT_COL].apply(clean_text)
    df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)
    print(f"Loaded {len(df)} rows (dropped {before - len(df)} unmapped).")
    return df


def composite_label(df):
    """Encode (sentiment, intent) as a single integer for stratified splitting."""
    return df[SENTIMENT_COL] * 10 + df[INTENT_COL]


df = load_and_clean(FULL_CSV)

print("\nSentiment distribution:")
for lbl, cnt in df[SENTIMENT_COL].value_counts().sort_index().items():
    print(f"  [{lbl}] {SENTIMENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df)*100:.1f}%)")
print("\nIntent distribution:")
for lbl, cnt in df[INTENT_COL].value_counts().sort_index().items():
    print(f"  [{lbl}] {INTENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df)*100:.1f}%)")

df.head()

Loaded 5881 rows (dropped 0 unmapped).

Sentiment distribution:
  [0] positive       397 (6.8%)
  [1] negative      1534 (26.1%)
  [2] neutral       3888 (66.1%)
  [3] conflict        62 (1.1%)

Intent distribution:
  [0] suggestion    3208 (54.5%)
  [1] complaint      541 (9.2%)
  [2] inquiry        448 (7.6%)
  [3] feedback       785 (13.3%)
  [4] greeting       593 (10.1%)
  [5] other          306 (5.2%)


,text,sentiment_label,intent_label,combined_label,nli_conf,conf_tier,nli_rule_agree,sample_weight
0,Ammage Adarayta❤️Eka Dawasak Madi Neda❤️🙏❤️,1,0,Negative_inquiry,0.3559,rule,True,1.0
1,We need IPL Champions leak data offers ..pleas...,2,0,Neutral_inquiry,0.7913,high,False,1.0
2,chandimal.. uuu thama mulu tem ekama kaaa gaha...,0,0,Positive_inquiry,0.4922,rule,False,1.0
3,Lebsack 49k dammama eka dawasak wath be,1,0,Negative_inquiry,0.8232,rule,True,1.0
4,eth anith kattiya sathiyak withara online inna...,2,5,Neutral_other,0.6918,mid,False,0.5


## 5 — Dataset & DataLoader

In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

from huggingface_hub import login
login(token=secret_value_0)

In [7]:
class JointDataset(Dataset):
    def __init__(self, texts, sent_lbls, intent_lbls, tokenizer):
        self.texts       = texts
        self.sent_lbls   = sent_lbls
        self.intent_lbls = intent_lbls
        self.tok         = tokenizer

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(str(self.texts[idx]), max_length=MAX_LEN,
                       padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids":       enc["input_ids"].squeeze(0),
            "attention_mask":  enc["attention_mask"].squeeze(0),
            "sentiment_label": torch.tensor(self.sent_lbls[idx],   dtype=torch.long),
            "intent_label":    torch.tensor(self.intent_lbls[idx], dtype=torch.long),
        }


def make_loader(df, tok, shuffle=False):
    ds = JointDataset(df[TEXT_COL].tolist(),
                      df[SENTIMENT_COL].tolist(),
                      df[INTENT_COL].tolist(), tok)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=True)


print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
print(f"Tokenizer ready  (vocab size: {tokenizer.vocab_size:,})")

Loading tokenizer...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready  (vocab size: 250,002)


## 6 — Model Architecture

In [8]:
class JointModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(BACKBONE)
        h = self.encoder.config.hidden_size
        # Simpler head for cleaner sentiment labels
        self.sent_head = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(h, NUM_SENTIMENTS))
        # Deeper head for noisier intent labels
        self.intent_head = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(h, 256),
            nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(256, NUM_INTENTS))

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.sent_head(cls), self.intent_head(cls)

    def param_groups(self, lr, wd):
        """Separate param groups: encoder at lr, heads at lr*10; no decay on bias/LN."""
        nd  = ["bias", "LayerNorm.weight"]
        enc = list(self.encoder.named_parameters())
        hd  = (list(self.sent_head.named_parameters()) +
               list(self.intent_head.named_parameters()))
        return [
            {"params": [p for n,p in enc if not any(d in n for d in nd)], "lr": lr,    "weight_decay": wd},
            {"params": [p for n,p in enc if     any(d in n for d in nd)], "lr": lr,    "weight_decay": 0},
            {"params": [p for n,p in hd  if not any(d in n for d in nd)], "lr": lr*10, "weight_decay": wd},
            {"params": [p for n,p in hd  if     any(d in n for d in nd)], "lr": lr*10, "weight_decay": 0},
        ]


# Sanity-check forward pass with a dummy batch
print("Loading backbone for forward-pass check (~1.1 GB on first run)...")
_m = JointModel().to(DEVICE)
_d = torch.zeros(2, MAX_LEN, dtype=torch.long).to(DEVICE)
_k = torch.ones(2, MAX_LEN, dtype=torch.long).to(DEVICE)
_s, _i = _m(_d, _k)
print(f"Forward pass OK  → sentiment {tuple(_s.shape)}  intent {tuple(_i.shape)}")
print(f"Total parameters: {sum(p.numel() for p in _m.parameters()):,}")
del _m, _d, _k, _s, _i

Loading backbone for forward-pass check (~1.1 GB on first run)...


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Forward pass OK  → sentiment (2, 4)  intent (2, 6)
Total parameters: 278,245,130


## 7 — Loss Function

In [9]:
# Label-smoothed cross-entropy for each task
sc = nn.CrossEntropyLoss(label_smoothing=SENT_SMOOTH)
ic = nn.CrossEntropyLoss(label_smoothing=INTENT_SMOOTH)

def jloss(sl, il, slbl, ilbl):
    """Weighted sum of sentiment + intent losses (ALPHA controls the balance)."""
    ls = sc(sl, slbl)
    li = ic(il, ilbl)
    return ALPHA * ls + (1 - ALPHA) * li, ls, li

print(f"Joint loss = {ALPHA}×sent_loss + {1-ALPHA}×intent_loss  (label smoothing: sent={SENT_SMOOTH}, intent={INTENT_SMOOTH})")

Joint loss = 0.5×sent_loss + 0.5×intent_loss  (label smoothing: sent=0.1, intent=0.15)


## 8 — Sampling Helpers

Resampling operates on the **composite label** so both task dimensions are balanced together.
Only ever called on the training split — the validation fold is never touched.

In [10]:
def apply_sampling(df_train, strategy):
    """
    Resample df_train using the composite (sentiment × intent) label.
    Returns a new DataFrame with resampled rows.
    IMPORTANT: Only apply to training data — never to validation.
    """
    if strategy == "none":
        return df_train

    comp = (df_train[SENTIMENT_COL] * 10 + df_train[INTENT_COL]).values.reshape(-1, 1)
    idx  = np.arange(len(df_train)).reshape(-1, 1)

    if strategy == "oversample":
        sampler = RandomOverSampler(random_state=SEED)
        idx_res, _ = sampler.fit_resample(idx, comp)

    elif strategy == "undersample":
        sampler = RandomUnderSampler(random_state=SEED)
        idx_res, _ = sampler.fit_resample(idx, comp)

    elif strategy == "both":
        # Step 1: undersample majority down to 3× minority count
        minority_count  = pd.Series(comp.ravel()).value_counts().min()
        target_majority = min(3 * minority_count,
                              pd.Series(comp.ravel()).value_counts().max())
        sampling_dict_under = {
            cls: min(cnt, target_majority)
            for cls, cnt in pd.Series(comp.ravel()).value_counts().items()
        }
        u_sampler = RandomUnderSampler(sampling_strategy=sampling_dict_under,
                                       random_state=SEED)
        idx_mid, comp_mid = u_sampler.fit_resample(idx, comp)
        # Step 2: oversample minority up to new majority count
        o_sampler = RandomOverSampler(random_state=SEED)
        idx_res, _ = o_sampler.fit_resample(idx_mid, comp_mid)

    else:
        raise ValueError(f"Unknown sampling strategy: '{strategy}'. "
                          "Choose from: oversample | undersample | both | none")

    return df_train.iloc[idx_res.ravel()].reset_index(drop=True)


print("apply_sampling() ready")

apply_sampling() ready


## 9 — Training & Evaluation Loops

In [11]:
def train_epoch(model, loader, optimizer, scheduler, epoch, n_epochs):
    model.train()
    tot = tsl = til = 0.0
    pbar = tqdm(loader, desc=f"  Train {epoch}/{n_epochs}", leave=False,
                unit="batch", dynamic_ncols=True)
    for i, b in enumerate(pbar, 1):
        ids   = b["input_ids"].to(DEVICE)
        mask  = b["attention_mask"].to(DEVICE)
        s_lbl = b["sentiment_label"].to(DEVICE)
        i_lbl = b["intent_label"].to(DEVICE)
        sl, il = model(ids, mask)
        loss, ls, li = jloss(sl, il, s_lbl, i_lbl)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        tot += loss.item(); tsl += ls.item(); til += li.item()
        pbar.set_postfix(loss=f"{tot/i:.4f}", s_loss=f"{tsl/i:.4f}", i_loss=f"{til/i:.4f}")
    n = len(loader)
    return tot/n, tsl/n, til/n


@torch.no_grad()
def eval_epoch(model, loader, epoch, n_epochs):
    model.eval()
    tot = cs = ci = jc = total = 0
    all_sp, all_ip, all_sl, all_il = [], [], [], []
    pbar = tqdm(loader, desc=f"  Val   {epoch}/{n_epochs}", leave=False,
                unit="batch", dynamic_ncols=True)
    for b in pbar:
        ids   = b["input_ids"].to(DEVICE)
        mask  = b["attention_mask"].to(DEVICE)
        s_lbl = b["sentiment_label"].to(DEVICE)
        i_lbl = b["intent_label"].to(DEVICE)
        sl, il = model(ids, mask)
        loss, _, _ = jloss(sl, il, s_lbl, i_lbl)
        tot += loss.item()
        sp = sl.argmax(1); ip = il.argmax(1)
        cs    += (sp == s_lbl).sum().item()
        ci    += (ip == i_lbl).sum().item()
        jc    += ((sp == s_lbl) & (ip == i_lbl)).sum().item()
        total += s_lbl.size(0)
        all_sp.extend(sp.cpu().tolist()); all_ip.extend(ip.cpu().tolist())
        all_sl.extend(s_lbl.cpu().tolist()); all_il.extend(i_lbl.cpu().tolist())
        pbar.set_postfix(loss=f"{tot/max(1, pbar.n):.4f}",
                         s_acc=f"{cs/total:.4f}", i_acc=f"{ci/total:.4f}")
    return {
        "loss":   tot / len(loader),
        "s_acc":  cs / total,
        "i_acc":  ci / total,
        "j_acc":  jc / total,
        "s_f1":   f1_score(all_sl, all_sp, average="macro", zero_division=0),
        "i_f1":   f1_score(all_il, all_ip, average="macro", zero_division=0),
        "all_sp": all_sp, "all_ip": all_ip,
        "all_sl": all_sl, "all_il": all_il,
    }


print("train_epoch() and eval_epoch() ready")

train_epoch() and eval_epoch() ready


## 10 — Cross-Validation Loop

This is the main training loop. For each fold it:
1. Splits data via `StratifiedKFold` on the composite label
2. Applies sampling to the training split only
3. Trains a fresh model with early stopping
4. Records best validation metrics

In [12]:
import warnings

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [13]:
def run_cv():
    comp_labels = composite_label(df).values
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_results = []
    fold_pbar = tqdm(enumerate(skf.split(np.zeros(len(df)), comp_labels)),
                     total=N_FOLDS, desc="Folds", unit="fold", dynamic_ncols=True)

    for fold, (train_idx, val_idx) in fold_pbar:
        print(f"\n{'='*60}")
        print(f"  FOLD {fold+1}/{N_FOLDS}")
        print(f"{'='*60}")

        df_train_raw = df.iloc[train_idx].reset_index(drop=True)
        df_val       = df.iloc[val_idx].reset_index(drop=True)

        # ── Class distribution BEFORE sampling ────────────────────────────
        print(f"\n[Fold {fold+1}] Train size before sampling: {len(df_train_raw)}")
        print("  Sentiment distribution (train):")
        for lbl, cnt in df_train_raw[SENTIMENT_COL].value_counts().sort_index().items():
            print(f"    [{lbl}] {SENTIMENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df_train_raw)*100:.1f}%)")
        print("  Intent distribution (train):")
        for lbl, cnt in df_train_raw[INTENT_COL].value_counts().sort_index().items():
            print(f"    [{lbl}] {INTENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df_train_raw)*100:.1f}%)")

        # ── Apply sampling (training data ONLY) ───────────────────────────
        df_train = apply_sampling(df_train_raw, SAMPLING_STRATEGY)
        print(f"\n[Fold {fold+1}] Train size after  sampling: {len(df_train)}")
        print("  Sentiment distribution (after sampling):")
        for lbl, cnt in df_train[SENTIMENT_COL].value_counts().sort_index().items():
            print(f"    [{lbl}] {SENTIMENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df_train)*100:.1f}%)")
        print("  Intent distribution (after sampling):")
        for lbl, cnt in df_train[INTENT_COL].value_counts().sort_index().items():
            print(f"    [{lbl}] {INTENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df_train)*100:.1f}%)")

        train_loader = make_loader(df_train, tokenizer, shuffle=True)
        val_loader   = make_loader(df_val,   tokenizer)

        # ── Fresh model per fold ──────────────────────────────────────────
        set_seed(SEED + fold)
        model     = JointModel().to(DEVICE)
        optimizer = torch.optim.AdamW(model.param_groups(LR, WEIGHT_DECAY))
        total_steps = len(train_loader) * EPOCHS
        scheduler   = get_linear_schedule_with_warmup(
            optimizer, int(total_steps * WARMUP_RATIO), total_steps)

        best_loss = float("inf"); pat = 0; best_metrics = None

        for epoch in range(1, EPOCHS + 1):
            tl, tsl, til = train_epoch(model, train_loader, optimizer, scheduler, epoch, EPOCHS)
            vm = eval_epoch(model, val_loader, epoch, EPOCHS)
            print(f"  Epoch {epoch}/{EPOCHS}  "
                  f"train_loss={tl:.4f}  val_loss={vm['loss']:.4f}  "
                  f"s_acc={vm['s_acc']:.4f}  i_acc={vm['i_acc']:.4f}  "
                  f"j_acc={vm['j_acc']:.4f}  s_f1={vm['s_f1']:.4f}  i_f1={vm['i_f1']:.4f}")
            if vm["loss"] < best_loss:
                best_loss = vm["loss"]; pat = 0; best_metrics = vm
                torch.save({
                    "fold": fold,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_metrics": vm,
                }, f"/kaggle/working/best_model_fold{fold+1}.pt")
                print("    ↳ best checkpoint saved")
            else:
                pat += 1
                if pat >= PATIENCE:
                    print(f"  Early stopping at epoch {epoch}"); break

        fold_results.append(best_metrics)
        fold_pbar.set_postfix(j_acc=f"{best_metrics['j_acc']:.4f}",
                              s_f1=f"{best_metrics['s_f1']:.4f}",
                              i_f1=f"{best_metrics['i_f1']:.4f}")
        print(f"\n[Fold {fold+1}] Best val  → "
              f"j_acc={best_metrics['j_acc']:.4f}  "
              f"s_f1={best_metrics['s_f1']:.4f}  "
              f"i_f1={best_metrics['i_f1']:.4f}")

    return fold_results


fold_results = run_cv()

Folds:   0%|          | 0/2 [00:00<?, ?fold/s]


  FOLD 1/2

[Fold 1] Train size before sampling: 2940
  Sentiment distribution (train):
    [0] positive       198 (6.7%)
    [1] negative       767 (26.1%)
    [2] neutral       1943 (66.1%)
    [3] conflict        32 (1.1%)
  Intent distribution (train):
    [0] suggestion    1604 (54.6%)
    [1] complaint      271 (9.2%)
    [2] inquiry        224 (7.6%)
    [3] feedback       392 (13.3%)
    [4] greeting       296 (10.1%)
    [5] other          153 (5.2%)

[Fold 1] Train size after  sampling: 28200
  Sentiment distribution (after sampling):
    [0] positive      7050 (25.0%)
    [1] negative      7050 (25.0%)
    [2] neutral       7050 (25.0%)
    [3] conflict      7050 (25.0%)
  Intent distribution (after sampling):
    [0] suggestion    4700 (16.7%)
    [1] complaint     4700 (16.7%)
    [2] inquiry       4700 (16.7%)
    [3] feedback      4700 (16.7%)
    [4] greeting      4700 (16.7%)
    [5] other         4700 (16.7%)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

  Train 1/3: 100%|██████████| 882/882 [10:23<00:00,  1.69batch/s, i_loss=1.0775, loss=0.9219, s_loss=0.7663]
                                                                                                            
  Val   1/3: 100%|██████████| 92/92 [00:18<00:00,  5.26batch/s, i_acc=0.7834, loss=1.0215, s_acc=0.7212]
                                                                                                        

  Epoch 1/3  train_loss=0.9219  val_loss=1.0104  s_acc=0.7212  i_acc=0.7834  j_acc=0.5675  s_f1=0.5210  i_f1=0.6868
    ↳ best checkpoint saved



  Train 2/3: 100%|██████████| 882/882 [10:26<00:00,  1.73batch/s, i_loss=0.6340, loss=0.5257, s_loss=0.4175]
                                                                                                            
  Val   2/3: 100%|██████████| 92/92 [00:17<00:00,  5.27batch/s, i_acc=0.8120, loss=0.9856, s_acc=0.7416]
                                                                                                        

... (10 outputs left out)


  Train 3/3: 100%|██████████| 882/882 [10:26<00:00,  1.41batch/s, i_loss=0.6074, loss=0.4946, s_loss=0.3819]
                                                                                                            
  Val   3/3: 100%|██████████| 92/92 [00:17<00:00,  5.27batch/s, i_acc=0.8061, loss=0.9826, s_acc=0.7497]
                                                                                                        

  Epoch 3/3  train_loss=0.4946  val_loss=0.9719  s_acc=0.7497  i_acc=0.8061  j_acc=0.6163  s_f1=0.5349  i_f1=0.7063


Folds: 100%|██████████| 2/2 [1:04:40<00:00, 1940.09s/fold, i_f1=0.7063, j_acc=0.6163, s_f1=0.5349]

    ↳ best checkpoint saved

[Fold 2] Best val  → j_acc=0.6163  s_f1=0.5349  i_f1=0.7063


## 11 — Cross-Validation Summary

In [14]:
print(f"{'='*60}")
print("  CROSS-VALIDATION SUMMARY")
print(f"  Sampling strategy : {SAMPLING_STRATEGY}")
print(f"  Folds             : {N_FOLDS}")
print(f"{'='*60}")

for key, name in [("j_acc","Joint Acc"), ("s_acc","Sent Acc"),
                  ("i_acc","Intent Acc"), ("s_f1","Sent Macro-F1"),
                  ("i_f1","Intent Macro-F1")]:
    vals = [r[key] for r in fold_results]
    print(f"  {name:<18}: {np.mean(vals):.4f} ± {np.std(vals):.4f}  "
          f"(folds: {[round(v,4) for v in vals]})")

  CROSS-VALIDATION SUMMARY
  Sampling strategy : oversample
  Folds             : 2
  Joint Acc         : 0.6186 ± 0.0023  (folds: [0.6209, 0.6163])
  Sent Acc          : 0.7536 ± 0.0040  (folds: [0.7576, 0.7497])
  Intent Acc        : 0.8113 ± 0.0051  (folds: [0.8164, 0.8061])
  Sent Macro-F1     : 0.5337 ± 0.0012  (folds: [0.5324, 0.5349])
  Intent Macro-F1   : 0.7114 ± 0.0050  (folds: [0.7164, 0.7063])


## 12 — Detailed Classification Reports (Last Fold)

In [15]:
last = fold_results[-1]
present_s = sorted(set(last["all_sl"]))
present_i = sorted(set(last["all_il"]))

print("[Last fold] Sentiment classification report:")
print(classification_report(last["all_sl"], last["all_sp"],
                            labels=present_s,
                            target_names=[SENTIMENT_CLASSES[i] for i in present_s],
                            digits=4, zero_division=0))

print("\n[Last fold] Intent classification report:")
print(classification_report(last["all_il"], last["all_ip"],
                            labels=present_i,
                            target_names=[INTENT_CLASSES[i] for i in present_i],
                            digits=4, zero_division=0))

[Last fold] Sentiment classification report:
              precision    recall  f1-score   support

    positive     0.5525    0.5051    0.5277       198
    negative     0.6086    0.7379    0.6671       767
     neutral     0.8462    0.7900    0.8171      1943
    conflict     0.2000    0.0938    0.1277        32

    accuracy                         0.7497      2940
   macro avg     0.5518    0.5317    0.5349      2940
weighted avg     0.7574    0.7497    0.7510      2940


[Last fold] Intent classification report:
              precision    recall  f1-score   support

  suggestion     0.9237    0.8978    0.9105      1604
   complaint     0.6744    0.6421    0.6578       271
     inquiry     0.6507    0.6652    0.6578       224
    feedback     0.6098    0.7015    0.6524       392
    greeting     0.8609    0.8784    0.8696       296
       other     0.5106    0.4706    0.4898       153

    accuracy                         0.8061      2940
   macro avg     0.7050    0.7092    0.7063

In [16]:
# ── 13 — Load Best Model for Inference ───────────────────────────────────────

# Pick the fold with the highest joint accuracy
best_fold_idx = max(range(len(fold_results)), key=lambda i: fold_results[i]["j_acc"])
best_ckpt_path = f"/kaggle/working/best_model_fold{best_fold_idx+1}.pt"
print(f"Best fold: {best_fold_idx+1}  →  loading {best_ckpt_path}")

inference_model = JointModel().to(DEVICE)
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
inference_model.load_state_dict(ckpt["model_state_dict"])
inference_model.eval()
print(f"Loaded checkpoint from fold {ckpt['fold']+1}, epoch {ckpt['epoch']}")
print(f"Val metrics at save: {ckpt['val_metrics']}")


def predict(texts: list[str]) -> list[dict]:
    enc = tokenizer(
        texts,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        s_logits, i_logits = inference_model(input_ids, attention_mask)

    s_preds = s_logits.argmax(dim=-1).cpu().tolist()
    i_preds = i_logits.argmax(dim=-1).cpu().tolist()

    return [
        {
            "text":      t,
            "sentiment": SENTIMENT_CLASSES[s],
            "intent":    INTENT_CLASSES[i],
        }
        for t, s, i in zip(texts, s_preds, i_preds)
    ]


# Quick smoke-test
samples = ["උඹ හරිම හොඳයි", "මට ගෙදර යන්න ඕනි"]
for result in predict(samples):
    print(result)

Best fold: 1  →  loading /kaggle/working/best_model_fold1.pt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded checkpoint from fold 1, epoch 3
Val metrics at save: {'loss': 0.9679530146329299, 's_acc': 0.7575654539272356, 'i_acc': 0.8163889833390003, 'j_acc': 0.6208772526351581, 's_f1': 0.5324156779125611, 'i_f1': 0.716418331709408, 'all_sp': [2, 1, 1, 2, 2, 2, 0, 1, 2, 2, 1, 2, 0, 2, 1, 2, 1, 2, 0, 2, 2, 2, 1, 2, 2, 2, 2, 2, 0, 2, 2, 2, 1, 1, 2, 1, 1, 1, 2, 0, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 0, 1, 2, 2, 1, 1, 2, 3, 0, 2, 2, 1, 2, 2, 2, 0, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 1, 2, 2, 1, 2, 1, 2, 2, 2, 2, 0, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 1, 2, 2, 0, 1, 2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 1, 0, 2, 2, 2, 0, 1, 2, 1, 1, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 2, 1, 1, 1, 0, 1, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 1, 2, 2, 1, 2, 2, 2, 2, 1, 2, 2, 1, 2, 1, 1, 2, 1, 2, 2, 1, 2, 1, 1, 2, 2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 0, 2, 2, 1, 1, 1, 1, 2, 1, 1, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 0, 1, 2, 2, 2, 2, 1, 2, 1,

In [17]:
samples = ["උඹ හරිම හොඳයි", "මට ගෙදර යන්න ඕනි", "Mage yaka awussa ganna epa miniho!"]
for result in predict(samples):
    print(result)

{'text': 'උඹ හරිම හොඳයි', 'sentiment': 'positive', 'intent': 'greeting'}
{'text': 'මට ගෙදර යන්න ඕනි', 'sentiment': 'neutral', 'intent': 'suggestion'}
{'text': 'Mage yaka awussa ganna epa miniho!', 'sentiment': 'negative', 'intent': 'suggestion'}


In [19]:
import os
os.makedirs("/kaggle/working/best_model", exist_ok=True)

# Save using the same checkpoint structure the notebook uses
torch.save({
    "model_state_dict": inference_model.state_dict(),
    "fold":             ckpt["fold"],
    "epoch":            ckpt["epoch"],
    "val_metrics":      ckpt["val_metrics"],
}, "/kaggle/working/best_model/best_model.pt")

# Save tokenizer too
tokenizer.save_pretrained("/kaggle/working/best_model/")

print("Saved! Fold:", ckpt["fold"]+1, "| Epoch:", ckpt["epoch"])
print("Val metrics:", ckpt["val_metrics"])

Saved! Fold: 1 | Epoch: 3
Val metrics: {'loss': 0.9679530146329299, 's_acc': 0.7575654539272356, 'i_acc': 0.8163889833390003, 'j_acc': 0.6208772526351581, 's_f1': 0.5324156779125611, 'i_f1': 0.716418331709408, 'all_sp': [2, 1, 1, 2, 2, 2, 0, 1, 2, 2, 1, 2, 0, 2, 1, 2, 1, 2, 0, 2, 2, 2, 1, 2, 2, 2, 2, 2, 0, 2, 2, 2, 1, 1, 2, 1, 1, 1, 2, 0, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 0, 1, 2, 2, 1, 1, 2, 3, 0, 2, 2, 1, 2, 2, 2, 0, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 1, 2, 2, 1, 2, 1, 2, 2, 2, 2, 0, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 1, 2, 2, 0, 1, 2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 1, 0, 2, 2, 2, 0, 1, 2, 1, 1, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 2, 1, 1, 1, 0, 1, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 1, 2, 2, 1, 2, 2, 2, 2, 1, 2, 2, 1, 2, 1, 1, 2, 1, 2, 2, 1, 2, 1, 1, 2, 2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 0, 2, 2, 1, 1, 1, 1, 2, 1, 1, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 0, 1, 2, 2, 2, 2, 1, 2, 1, 2, 1, 1, 1, 0, 1, 2,

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

# ── Config (must match training) ──────────────────────────────────────────────
BACKBONE        = "xlm-roberta-base"
MAX_LEN         = 128
DROPOUT         = 0.3
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SENTIMENT_CLASSES = ["positive", "negative", "neutral", "conflict"]
INTENT_CLASSES    = ["suggestion", "complaint", "inquiry", "feedback", "greeting", "other"]
NUM_SENTIMENTS    = len(SENTIMENT_CLASSES)
NUM_INTENTS       = len(INTENT_CLASSES)

# ── Model definition (must match training) ────────────────────────────────────
class JointModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(BACKBONE)
        hidden = self.encoder.config.hidden_size
        self.sent_head   = nn.Linear(hidden, NUM_SENTIMENTS)
        self.intent_head = nn.Linear(hidden, NUM_INTENTS)
        self.dropout     = nn.Dropout(DROPOUT)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        return self.sent_head(cls), self.intent_head(cls)

# ── Load ──────────────────────────────────────────────────────────────────────
MODEL_DIR = "/kaggle/working/best_model"   # change to your local path if needed

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = JointModel().to(DEVICE)
ckpt  = torch.load(f"{MODEL_DIR}/best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"Loaded — fold {ckpt['fold']+1}, epoch {ckpt['epoch']}")
print(f"Val metrics: {ckpt['val_metrics']}")

# ── Inference ─────────────────────────────────────────────────────────────────
def predict(texts: list[str]) -> list[dict]:
    enc = tokenizer(
        texts,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        s_logits, i_logits = model(input_ids, attention_mask)

    s_preds = s_logits.argmax(dim=-1).cpu().tolist()
    i_preds = i_logits.argmax(dim=-1).cpu().tolist()

    return [
        {
            "text":      t,
            "sentiment": SENTIMENT_CLASSES[s],
            "intent":    INTENT_CLASSES[i],
        }
        for t, s, i in zip(texts, s_preds, i_preds)
    ]

# ── Test ──────────────────────────────────────────────────────────────────────
samples = ["උඹ හරිම හොඳයි", "මට ගෙදර යන්න ඕනි", "Mage yaka awussa ganna epa miniho!"]
for result in predict(samples):
    print(result)